In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.9 The Ideal Fermi Gas at T = 0: The Fermi Sea and the Stiffness of Matter

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.9",
    title="The Ideal Fermi Gas at T = 0: The Fermi Sea and the Stiffness of Matter",
    blurb="A classical gas at absolute zero lies down; a fermion gas cannot. Forbidden from "
    "sharing states, the electrons of a metal stack into a filled sea whose surface "
    "sits tens of thousands of kelvin above the room around it — so a copper wire on "
    "a bench is, to its electrons, a system at absolute zero, humming with a million "
    "metres per second of purely quantum motion. The sea pushes back when squeezed: "
    "we compute that push, check it against the measured stiffness of real metals — "
    "the alkalis agree to ten percent with nothing fitted — and hear it in the speed "
    "of sound.",
    difficulty="advanced",
    estimate="185–225 min",
)

## Notebook overview

Movement III opens where the degeneracy map of [§7.8](classical-limit-thermal-wavelength.ipynb) pointed: dense and light. Copper's conduction
electrons sat four decades above the classical boundary ($n\lambda_T^3 = 6.8\times10^3$), and
this notebook builds the state they actually occupy. At absolute zero a classical gas does
nothing — zero energy, zero pressure, no objection to being squeezed. A gas of fermions cannot
afford such rest: the Pauli principle ([§6.20](../06-quantum-mechanics/identical-particles.ipynb)) forbids more than one particle per state (two with
spin), so the ground state is not "everything in the lowest level" but a **filled sea** — every
state occupied up to a sharp surface, the Fermi energy, with nothing but empty states above it.

The course's move is to build the sea *literally*. We take the particle-in-a-box levels of [§7.3](statistical-toolkit.ipynb), sort
them, spin-double them, and stack twenty thousand fermions from the bottom. Two honest phenomena
appear on the way to the continuum: the highest occupied energy $\varepsilon_F(N)$ climbs a
**staircase** (degenerate box shells filling one by one, finite-size shell structure presented
as physics rather than noise; its grand descendants organize nuclei and metal clusters), and
the energy per particle converges to the continuum $\tfrac35\varepsilon_F$ *from above*, the
occupied-side cousin of the Weyl surface deficit of [§7.3](statistical-toolkit.ipynb). The continuum machinery then delivers the
scales, and for real metals the numbers rearrange one's intuition: copper's
$\varepsilon_F = 7.0$ eV means $T_F = 8.2\times10^4$ K, so **room temperature is 0.4% of the
Fermi temperature** — a metal on a lab bench is, for its electrons, a system at effectively
absolute zero, each electron carrying about a hundred times the classical thermal energy as pure
Pauli zero-point motion, cruising at $10^6$ m/s in the coldest possible state.

The centerpiece is what the sea does when squeezed: **degeneracy pressure**, $P =
\tfrac25 n\varepsilon_F \approx 38$ GPa in copper at $T = 0$ — the statistical repulsion of [§7.8](classical-limit-thermal-wavelength.ipynb) grown
into the stiffness of matter — and the claim is put to laboratory data with *nothing fitted*:
the free-electron bulk modulus $B = \tfrac23 n\varepsilon_F$ lands within 10% of the measured
compressibilities of the alkali metals (potassium to 2%), while sodium's overshoot and copper's
factor-two shortfall are taught as the model's honest boundary. A stretch exercise turns
stiffness into sound (the Bohm–Staver speed predicts sodium's 3.0 km/s against the measured
3.2: one hears the exclusion principle in a struck metal bar), and the closing computation
locates the movement's summit in advance: the density at which $v_F$ reaches $c$ sits exactly at
white-dwarf core densities, where the pressure law bends and stars begin to lose the argument
with gravity. That cliff belongs to [§7.11](white-dwarfs-chandrasekhar.ipynb); this notebook walks to its edge. Everything
finite-temperature — $\mu(T)$, the Sommerfeld corrections, the linear heat capacity, Pauli
paramagnetism — is deliberately deferred to [§7.10](fermi-gas-finite-temperature.ipynb).

> **Conventions (this notebook).** The spin factor 2 is included throughout (and the literal
> filling spin-doubles the level list *before* slicing — doubling after silently rescales every
> energy by $2^{2/3}$). Box-model results are labelled finite-size, continuum results continuum.
> All data comparisons run in SI with CODATA constants in Setup; conduction-electron
> densities and measured bulk moduli follow Ashcroft & Mermin (Tables 1.1 and 2.1 lineage), with
> the free-electron counts $Z = 1$ for the alkalis and copper's single 4s electron.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the DOS integral inverted back to $N$ at eight digits; the literal filling's
> $E/(N\varepsilon_F)$ marching $0.638 \to 0.620 \to 0.609$ onto the continuum $\tfrac35$;
> copper's $\varepsilon_F = 7.03$ eV; the $2^{-2/3}$ volume scaling and a finite-difference
> check of $P = -\partial E/\partial V$; the alkali modulus ratios ($1.02$, $0.91$, $0.97$)
> against handbook data; Bohm–Staver's $3011$ m/s for sodium; and the relativistic threshold
> $5.87\times10^{35}$ m⁻³ bracketed by white-dwarf densities. A ✓ is strong evidence; a ✗ is a
> prompt to *locate the discrepancy*.
>
> **Scope.** The ideal Fermi gas at exactly $T = 0$. The warm corrections are [§7.10](fermi-gas-finite-temperature.ipynb); the
> relativistic gas and the Chandrasekhar mass are [§7.11](white-dwarfs-chandrasekhar.ipynb) (only the threshold is located here);
> the sea meeting a lattice is [§7.12](bloch-theorem-band-structure.ipynb); degenerate doping returns in [§7.13](semiconductor-statistics.ipynb). Interacting electrons,
> screening, and Fermi-liquid theory are named horizons (Volume VIII territory). See Ashcroft &
> Mermin, *Solid State Physics* (Ch. 2); Pathria & Beale (Ch. 8); Kittel; Bohm & Staver (1951).
> Cross-reference [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (Pauli, the sea's foundation), [§7.7](bose-einstein-fermi-dirac.ipynb) (the step), [§7.8](classical-limit-thermal-wavelength.ipynb) (the degeneracy map
> and the fermionic virial), [§7.3](statistical-toolkit.ipynb) (the DOS, the counting, the Weyl deficit).

## Theory in brief

### The Fermi sea

At $T = 0$ the Fermi–Dirac step of [§7.7](bose-einstein-fermi-dirac.ipynb) is exact: every state below $\mu$ occupied, every state
above empty. The ground state of $N$ fermions is a filled sea, and its surface follows from
counting with the density of states of [§7.3](statistical-toolkit.ipynb) (spin factor 2 included):

```{math}
:label: eq-fermi-sea
N = \int_0^{\varepsilon_F} g(\varepsilon)\,d\varepsilon,
\quad
g(\varepsilon) = \frac{V}{2\pi^2}\left(\frac{2m}{\hbar^2}\right)^{3/2}\sqrt{\varepsilon}
\;\;\Longrightarrow\;\;
\varepsilon_F = \frac{\hbar^2}{2m}\big(3\pi^2 n\big)^{2/3},
\quad
k_F = \big(3\pi^2 n\big)^{1/3},
```

with $T_F = \varepsilon_F/k_B$ and $v_F = \hbar k_F/m$. In $k$-space the occupied states fill
the **Fermi sphere** of radius $k_F$; its boundary, the **Fermi surface**, is the single concept
that organizes all of metal physics — perfectly spherical here, reshaped by a lattice in [§7.12](bloch-theorem-band-structure.ipynb).

### Filling the sea literally

The course's standing move: before trusting the integral, do the count.

```{math}
:label: eq-literal-filling
\varepsilon_{n_xn_yn_z} \propto n_x^2 + n_y^2 + n_z^2,
\qquad
\text{sort} \to \text{spin-double} \to \text{occupy the } N \text{ lowest},
```

and two honest pieces of physics appear. The highest occupied energy $\varepsilon_F(N)$ climbs a
**staircase**: box shells are degenerate, and each fills completely before the next opens —
finite-size **shell structure**, the tabletop cousin of nuclear magic numbers and the abundance
spikes of metal clusters. And the energy per particle approaches the continuum from above,
$E/(N\varepsilon_F) = 0.638, 0.620, 0.609$ at $N = 10^2, 10^3, 2\times10^4$: the excess is the
surface (Weyl) correction of [§7.3](statistical-toolkit.ipynb), returning on the occupied side of the ledger.

### The 3/5 and the scales of real metals

In the continuum the mean kinetic energy per fermion is

```{math}
:label: eq-metal-scales
\langle\varepsilon\rangle
= \frac{\int_0^{\varepsilon_F}\varepsilon\,g(\varepsilon)\,d\varepsilon}
       {\int_0^{\varepsilon_F}g(\varepsilon)\,d\varepsilon}
= \frac{3}{5}\,\varepsilon_F,
```

and the numbers for real metals deserve slow reading. Copper: $\varepsilon_F = 7.03$ eV,
$T_F = 8.16\times10^4$ K, $v_F = 1.57\times10^6$ m/s. Sodium through cesium: $3.24$ down to
$1.59$ eV. Two consequences. First, $T_{\text{room}}/T_F = 0.4$–$1.6\%$: a metal is *always* in
the deep quantum regime — the $n\lambda^3 = 6.8\times10^3$ of [§7.8](classical-limit-thermal-wavelength.ipynb) said the same thing from the other
side — so this notebook's $T = 0$ idealization is accurate to parts in a thousand, and the
entire job of [§7.10](fermi-gas-finite-temperature.ipynb) is the small, Sommerfeld-shaped remainder. Second, each electron carries
$\tfrac35\times7$ eV $\approx 4$ eV of kinetic energy, roughly a hundred times the classical
$\tfrac32 k_BT$ at room temperature, as pure Pauli **zero-point motion**: the coldest copper
hums at a million metres per second.

### Degeneracy pressure: the stiffness of matter

At fixed $N$ the total energy scales as $E = \tfrac35 N\varepsilon_F \propto V^{-2/3}$, so the
sea pushes:

```{math}
:label: eq-degeneracy-pressure
P = -\left(\frac{\partial E}{\partial V}\right)_N = \frac{2}{3}\frac{E}{V}
= \frac{2}{5}\,n\,\varepsilon_F ,
```

pressure at absolute zero, with no interactions and no temperature — the grown-up form of the
fermionic virial excess of [§7.8](classical-limit-thermal-wavelength.ipynb). For copper's electron gas: $38$ GPa, four hundred thousand atmospheres
of outward Pauli push, balanced in the real metal by the electron–ion attraction. *This* is why
ordinary matter resists compression: to shrink the box, the sea must climb its own ladder.

### The bulk moduli of real metals: the data jewel

Differentiating once more gives a zero-parameter prediction from the electron density alone:

```{math}
:label: eq-bulk-modulus
B = -V\left(\frac{\partial P}{\partial V}\right) = \frac{5}{3}P = \frac{2}{3}\,n\,\varepsilon_F
\;\propto\; n^{5/3} .
```

Against the handbook: potassium $3.2$ predicted vs $3.1$ GPa measured (ratio $1.02$), cesium
$0.97$, rubidium $0.91$ — the alkalis, one loosely bound s-electron over a closed shell and the
free-electron poster children, land within 10% with *nothing fitted*. Sodium overshoots
($1.46$) and copper falls short by two ($0.46$): the electron–ion softening, the d-band, and
the lattice do real work the free gas cannot. A model validated *including* its boundary — the
course's standing epistemic standard.

### The Bohm–Staver sound speed

Sound in a metal is ions (mass $M$, valence $Z$) oscillating against the electron gas's spring.
At the jellium level the compressibility of the sea sets the restoring force and the ions supply
the inertia, and the assembly gives

```{math}
:label: eq-bohm-staver
v_s = v_F\,\sqrt{\frac{Z\,m_e}{3M}} ,
```

the **Bohm–Staver relation**. Predictions with no parameters: sodium $3011$ m/s against the
measured $\sim3200$; potassium $1867$ against $\sim2000$ — 5–7%. The longitudinal ring of a
struck metal bar is, to leading order, the Fermi velocity times a mass ratio: one *hears* the
exclusion principle. (Its proper home, the electron–phonon problem, is named and left
outward; Ashcroft & Mermin, *Solid State Physics*, Ch. 26, carries the derivation out in full.)

### The edge of the cliff

$v_F = \hbar(3\pi^2n)^{1/3}/m_e$ grows with density, and formally reaches $c$ at

```{math}
:label: eq-relativistic-threshold
n_{\text{rel}} = \frac{1}{3\pi^2}\left(\frac{m_ec}{\hbar}\right)^{3} = 5.87\times10^{35}\ \text{m}^{-3},
```

where the kinematics must turn relativistic: $\varepsilon(k)$ softens from $\hbar^2k^2/2m$
toward $\hbar ck$, and the pressure law from $n^{5/3}$ toward $n^{4/3}$. Locate the threshold
physically: a white dwarf's core at $\rho = 10^9$–$10^{10}$ kg/m³ (electron density
$n_e = \rho/2u$ for carbon/oxygen composition) *brackets* $n_{\text{rel}}$ — the stellar corpses
of [§7.11](white-dwarfs-chandrasekhar.ipynb) live exactly where the stiffness law bends, and on that bend hangs a maximum mass. The
cliff is located; [§7.11](white-dwarfs-chandrasekhar.ipynb) walks off it.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: spin factor 2 everywhere (the literal filling spin-doubles
# BEFORE slicing); SI with CODATA constants for every data comparison; conduction-
# electron densities and measured bulk moduli in the Ashcroft & Mermin lineage
# (Tables 1.1 / 2.1), Z = 1 free electrons for the alkalis and copper's 4s.
from scipy.constants import hbar as HBAR  # J s
from scipy.constants import m_e as M_ELECTRON  # kg
from scipy.constants import k as K_B  # J/K (exact)
from scipy.constants import eV as EV  # J (exact)
from scipy.constants import m_u as U_AMU  # atomic mass constant, kg
from scipy.constants import c as C_LIGHT  # m/s (exact)

# free-electron densities n (1/m^3), Ashcroft & Mermin Table 1.1 lineage
N_ELECTRON = {"Cu": 8.47e28, "Na": 2.65e28, "K": 1.40e28, "Rb": 1.15e28, "Cs": 0.91e28}
# measured bulk moduli (GPa), Ashcroft & Mermin Table 2.1 lineage
B_MEASURED = {"Na": 6.3, "K": 3.1, "Rb": 2.5, "Cs": 1.6, "Cu": 137.0}


def box_levels(nmax):
    """All particle-in-a-box orbital energies n_x^2 + n_y^2 + n_z^2, sorted (working units).

    The lattice enumeration of §7.3, reused for filling instead of counting: numpy.meshgrid
    over positive integers up to nmax per axis, energies in ground-scale units
    (E_1 = ħ^2 π^2/2mL^2 = 1), flattened and numpy.sort-ed. Adequacy rule: the filled
    sea must satisfy max(occupied energy) << nmax^2, else the grid truncates the sea
    invisibly — Exercise 2 checks it explicitly.

    Parameters
    ----------
    nmax : int
        Grid extent per axis (the enumeration holds nmax^3 orbitals).

    Returns
    -------
    numpy.ndarray
        Sorted orbital energies (spin NOT yet included — see fill_sea).
    """
    axes = [np.arange(1, nmax + 1)] * 3
    gx, gy, gz = np.meshgrid(*axes, indexing="ij", sparse=True)
    e = (gx.astype(float) ** 2 + gy.astype(float) ** 2 + gz.astype(float) ** 2).ravel()
    return np.sort(e)


def fill_sea(levels, N):
    """Occupy the N lowest spin-states of a sorted orbital list: the Fermi sea, literally.

    Spin-doubles the orbital energies with numpy.repeat(levels, 2) and slices the N
    lowest. ORDER MATTERS: doubling after slicing would fill N orbitals instead of
    N/2 and silently rescale every energy scale by 2^(2/3) — the trap is avoided
    here, once, on behalf of the whole notebook.

    Parameters
    ----------
    levels : numpy.ndarray
        Sorted orbital energies from box_levels.
    N : int
        Number of fermions to place (N ≤ 2·len(levels)).

    Returns
    -------
    tuple
        (occupied energies array of length N, ε_F(N) = the highest occupied energy).
    """
    spin_states = np.repeat(levels, 2)  # each orbital holds two spins — BEFORE slicing
    occupied = spin_states[:N]
    return occupied, float(occupied[-1])


def fermi_scales(n, m=M_ELECTRON):
    """The four Fermi scales (ε_F, k_F, T_F, v_F) from the density, in SI (eq-fermi-sea).

    ε_F = (ħ^2/2m)(3π^2 n)^(2/3) with the spin-2 counting already inside the 3π^2;
    k_F = (3π^2 n)^(1/3), T_F = ε_F/k_B, v_F = ħk_F/m. For copper these return
    7.03 eV, 8.2e4 K, 1.6e6 m/s — the numbers that make a room-temperature metal an
    absolute-zero system.

    Parameters
    ----------
    n : float
        Number density in 1/m^3.
    m : float, optional
        Fermion mass in kg (default: the electron).

    Returns
    -------
    dict
        {'eps_F': J, 'k_F': 1/m, 'T_F': K, 'v_F': m/s}.
    """
    k_F = (3.0 * np.pi**2 * n) ** (1.0 / 3.0)
    eps_F = HBAR**2 * k_F**2 / (2.0 * m)
    return {"eps_F": eps_F, "k_F": k_F, "T_F": eps_F / K_B, "v_F": HBAR * k_F / m}


def degeneracy_pressure(n, m=M_ELECTRON):
    """The T = 0 degeneracy pressure P = (2/5) n ε_F in Pa (eq-degeneracy-pressure).

    Pressure with no temperature and no interaction: E ∝ V^(−2/3) at fixed N, so
    P = (2/3)E/V. For copper's electron gas this is 38 GPa — the Pauli push that the
    electron–ion attraction balances in the real metal, and the stiffness the bulk
    modulus measures.

    Parameters
    ----------
    n : float
        Number density in 1/m^3.
    m : float, optional
        Fermion mass in kg (default: the electron).

    Returns
    -------
    float
        Pressure in Pa.
    """
    return 0.4 * n * fermi_scales(n, m)["eps_F"]


def bulk_modulus(n, m=M_ELECTRON):
    """The free-electron bulk modulus B = (5/3)P = (2/3) n ε_F in Pa (eq-bulk-modulus).

    A zero-parameter prediction of a metal's stiffness from its electron density and
    three constants of nature — Exercise 5 tests it against the measured moduli of
    five metals and reads both the ~10% alkali agreement and the honest failures.

    Parameters
    ----------
    n : float
        Number density in 1/m^3.
    m : float, optional
        Fermion mass in kg (default: the electron).

    Returns
    -------
    float
        Bulk modulus in Pa.
    """
    return (5.0 / 3.0) * degeneracy_pressure(n, m)


def bohm_staver(n, Z, M_ion):
    """The Bohm–Staver sound speed v_s = v_F √(Z m_e/3M) in m/s (eq-bohm-staver).

    Jellium sound: the electron sea supplies the spring (its T = 0 compressibility),
    the ions supply the inertia. No parameters — the speed of sound in a simple metal
    from its electron density, its valence, and its ion mass.

    Parameters
    ----------
    n : float
        Conduction-electron density in 1/m^3.
    Z : int
        Valence (electrons contributed per ion).
    M_ion : float
        Ion mass in kg.

    Returns
    -------
    float
        Longitudinal sound speed in m/s.
    """
    v_F = fermi_scales(n)["v_F"]
    return v_F * np.sqrt(Z * M_ELECTRON / (3.0 * M_ion))

## Exercise 1 — The sea from the step

The $T = 0$ Fermi function fills a sphere in $k$-space, and four scales fall out of its radius.
Cite {eq}`eq-fermi-sea`.

1. Take the $T \to 0$ limit of the $n_F$ of [§7.7](bose-einstein-fermi-dirac.ipynb) and state the filled-sea ground state: occupation 1
   below $\varepsilon_F$, 0 above.
2. Integrate the [§7.3](statistical-toolkit.ipynb) density of states (spin 2) to $N$ and invert:
   $\varepsilon_F = (\hbar^2/2m)(3\pi^2n)^{2/3}$, with $k_F$, $T_F$, $v_F$ defined.
3. Verify the inversion numerically with `scipy.integrate.quad`: the DOS integral up to the
   derived $\varepsilon_F$ returns the density to at least eight digits (copper's numbers, SI).
4. Name the Fermi surface (prose): the sphere's boundary in $k$-space, the single organizing
   concept of metal physics — here perfectly spherical, in [§7.12](bloch-theorem-band-structure.ipynb) reshaped by a lattice.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    n_from_quad,
    n_Cu,
    "the Fermi sea: the DOS integral inverted to ε_F returns the density",
    rtol=1e-8,
)
validate.close(
    scales_Cu["eps_F"] / EV,
    (HBAR**2 / (2 * M_ELECTRON)) * (3 * np.pi**2 * n_Cu) ** (2.0 / 3.0) / EV,
    "the closed form ε_F = (ħ²/2m)(3π²n)^(2/3), assembled independently",
    rtol=1e-12,
)

## Exercise 2 — Filling the sea, literally

Twenty thousand fermions, stacked by hand — with shell structure and the $\tfrac35$ emerging
from a sorted list. Cite {eq}`eq-literal-filling`, {eq}`eq-metal-scales`.

1. Build the box levels with `numpy.meshgrid`, flatten and `numpy.sort`, and spin-double with
   `numpy.repeat(levels, 2)` — noting the trap: doubling *after* slicing silently rescales every
   energy by $2^{2/3}$.
2. Use `fill_sea` to record $\varepsilon_F(N)$ and $E(N)/N$ up to $N = 2\times10^4$, checking
   the grid adequacy max(occupied) $\ll$ nmax².
3. Confirm $E/(N\varepsilon_F) = 0.638, 0.620, 0.609$ at $N = 10^2, 10^3, 2\times10^4$ —
   converging on $\tfrac35$ from above — and identify the excess as the surface (Weyl)
   deficit of [§7.3](statistical-toolkit.ipynb) on the occupied side.
4. Plot the $\varepsilon_F(N)$ staircase and read it (prose): degenerate box shells filling one
   at a time — finite-size shell structure, the tabletop cousin of nuclear magic numbers and
   metal-cluster stability.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [ratios[100], ratios[1000], ratios[20000]],
    [0.638, 0.620, 0.609],
    "the literal filling: E/(N ε_F) = 0.638 → 0.620 → 0.609, marching on the continuum 3/5",
    rtol=1e-2,
)
validate.check(
    ratios[100] > ratios[1000] > ratios[20000] > 0.6 and headroom < 0.5,
    "the approach is monotone from above (the Weyl deficit, occupied side) on an adequate grid",
    f"headroom {headroom:.3f}",
)

## Exercise 3 — Metals by the numbers

Five elements, four scales each — and a room-temperature metal revealed as a zero-temperature
system. Cite {eq}`eq-metal-scales`.

1. Verify the continuum $\langle\varepsilon\rangle = \tfrac35\varepsilon_F$ with
   `scipy.integrate.quad` on the DOS moments, then tabulate $\varepsilon_F$, $T_F$, $v_F$ for
   Cu, Na, K, Rb, Cs from their conduction-electron densities (`fermi_scales`, sources in
   Setup).
2. Confirm the verified values (Cu: $7.03$ eV, $8.16\times10^4$ K, $1.57\times10^6$ m/s; Na
   $3.24$, K $2.12$, Rb $1.86$, Cs $1.59$ eV) and compute $T_{\text{room}}/T_F$ for each.
3. Compare the mean kinetic energy $\tfrac35\varepsilon_F$ against the classical
   $\tfrac32k_BT$ at 300 K ($\sim$100×), and state what it is: Pauli zero-point motion.
4. Draw the license (prose): metals are *always* deeply degenerate (the
   $n\lambda^3 = 6.8\times10^3$ of [§7.8](classical-limit-thermal-wavelength.ipynb) re-read), the $T = 0$ idealization is accurate to $\sim10^{-3}$,
   and the task of [§7.10](fermi-gas-finite-temperature.ipynb) is precisely the small, Sommerfeld-shaped rest.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    mean_over_eF,
    0.6,
    "the continuum mean: ⟨ε⟩ = (3/5)ε_F by quadrature on the DOS moments",
    rtol=1e-10,
)
validate.close(
    [table_scales[m]["eps_F"] / EV for m in ("Cu", "Na", "K", "Rb", "Cs")],
    [7.03, 3.24, 2.12, 1.86, 1.59],
    "the scales of real metals: ε_F from the electron density alone",
    rtol=1e-2,
)
validate.check(
    all(300.0 / table_scales[m]["T_F"] < 0.017 for m in table_scales),
    "T_room/T_F ≤ 1.6% for every metal: a lab-bench metal is an absolute-zero system",
    f"Cu: {300.0 / table_scales['Cu']['T_F']:.2%}",
)

## Exercise 4 — Degeneracy pressure

A gas at absolute zero that pushes with four hundred thousand atmospheres. Cite
{eq}`eq-degeneracy-pressure`.

1. Derive $P = \tfrac23 E/V = \tfrac25 n\varepsilon_F$ from $E \propto V^{-2/3}$ at fixed $N$.
2. Verify the $V$-scaling by the literal filling (the occupation list is volume-independent, so
   the box ratio is exact) *and*, independently, check $P = -\partial E/\partial V$ against a
   central finite difference of the continuum $E(V)$ at copper's density.
3. Evaluate $P$ for copper ($38$ GPa at $T = 0$, about $4\times10^5$ atm) and state what
   balances it in a real metal (the electron–ion attraction).
4. Connect backward and forward (prose): this is the statistical repulsion of [§7.8](classical-limit-thermal-wavelength.ipynb) at full strength —
   the fermionic virial's $+n\lambda^3/2^{5/2}$ grown into the incompressibility of matter — and
   the same push will hold up a dead star in [§7.11](white-dwarfs-chandrasekhar.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [P_Cu / 1e9, E_ratio],
    [38.2, 2.0 ** (-2.0 / 3.0)],
    "degeneracy pressure: Pauli's 38 GPa push in copper, and the exact V^(−2/3) scaling",
    rtol=1e-2,
)
validate.close(
    P_findiff,
    P_Cu,
    "P = −∂E/∂V by central differences meets (2/5)nε_F: the exponent earned, not assumed",
    rtol=1e-6,
)

## Exercise 5 — The stiffness of real metals: a zero-parameter test

The bulk modulus from the electron density alone, against the handbook — triumph and honest
failure in one bar chart. Cite {eq}`eq-bulk-modulus`.

1. Derive $B = \tfrac53 P = \tfrac23 n\varepsilon_F \propto n^{5/3}$.
2. Use `bulk_modulus` to compute $B$ for Na, K, Rb, Cs, Cu and compare against the measured
   values ($6.3, 3.1, 2.5, 1.6, 137$ GPa; Ashcroft & Mermin lineage, stated in Setup).
3. Confirm the ratios ($1.46, 1.02, 0.91, 0.97, 0.46$) and plot the free-vs-measured bar chart.
4. Read the pattern (prose): the alkalis, with one nearly free s-electron, agree to $\sim$10% with
   *nothing* fitted; sodium's excess and copper's factor-two shortfall mark where the free gas
   ends (electron–ion softening; the d-band and lattice) — a model validated including its
   boundary, the course's standing epistemic standard.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    B_ratio["K"],
    1.02,
    "the compressibility of potassium from ε_F alone: a zero-parameter 2% agreement",
    rtol=3e-2,
)
validate.close(
    [B_ratio["Rb"], B_ratio["Cs"], B_ratio["Na"], B_ratio["Cu"]],
    [0.91, 0.97, 1.46, 0.46],
    "the full pattern: alkalis within 10%, sodium's overshoot, copper's honest factor two",
    rtol=3e-2,
)

## Exercise 6 — The sound of Pauli

The Bohm–Staver relation: a metal's sound speed from its Fermi velocity and a mass ratio. Cite
{eq}`eq-bohm-staver`.

1. Assemble $v_s = v_F\sqrt{Zm_e/3M}$ from the jellium picture — ions of mass $M$ riding the
   electron gas's spring, the derivation sketched at the compressibility level (its named home
   is the electron–phonon problem).
2. Use `bohm_staver` to predict $v_s$ for Na and K (valence 1, ion masses from the atomic
   weights).
3. Confirm $3011$ m/s vs measured $\sim3200$ (Na) and $1867$ vs $\sim2000$ (K) — 5–7% with no
   parameters.
4. Reflect (prose): the longitudinal ring of a struck sodium bar is, to leading order, the
   exclusion principle times $\sqrt{m_e/M}$ — quantum statistics audible across a room.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    v_s_Na,
    3011.0,
    "Bohm–Staver: the sound of the exclusion principle in sodium",
    rtol=2e-2,
)
validate.close(
    v_s_K,
    1867.0,
    "and in potassium: 5–7% of measurement with no parameters",
    rtol=2e-2,
)

## Exercise 7 — The edge of the cliff

Where the Fermi velocity meets the speed of light — and which objects in the universe live
there. Cite {eq}`eq-relativistic-threshold`.

1. Set $\hbar k_F/m_e = c$ and solve: $n_{\text{rel}} = (m_ec/\hbar)^3/3\pi^2$; evaluate.
2. Convert white-dwarf central densities $\rho = 10^9$ and $10^{10}$ kg/m³ to electron
   densities ($n_e = \rho/2u$ for C/O composition) and confirm they *bracket*
   $n_{\text{rel}}$.
3. State the consequence (one derivation step + prose): relativistic kinematics softens
   $\varepsilon(k)$ from $k^2/2m$ toward $ck$, hence $P$ from $n^{5/3}$ toward $n^{4/3}$ — the
   stiffness law bends exactly where the dead stars live.
4. Pose the question of [§7.11](white-dwarfs-chandrasekhar.ipynb) (prose): a star held up by a pressure that softens under compression is
   a star that can lose — above what mass does gravity win?

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    n_rel,
    5.87e35,
    "the relativistic threshold: v_F = c at (m_e c/ħ)³/3π²",
    rtol=1e-2,
)
validate.check(
    ratio_low < 1.0 < ratio_high and 0.4 < ratio_low and ratio_high < 6.0,
    "white-dwarf core densities bracket the threshold: the dead stars live on the bend",
    f"ratios {ratio_low:.2f} and {ratio_high:.2f}",
)

## Exercise 8 — The stiffness of the world

The classical gas met absolute zero by giving up — no energy, no pressure, no objection to being
squeezed. The fermion gas met it by building a sea. Everything in this notebook flowed from that
refusal: energies of electron-volts where classical physics offered hundredths, a
room-temperature wire that is effectively at absolute zero, thirty-eight gigapascals of outward
push in resting copper, the measured stiffness of potassium reproduced from its electron density
and three constants of nature, and the ring of a metal bar carrying the exclusion principle to
the ear. Volume V's gases pushed on walls because they were hot; this one pushes because it is
*full*.

It is worth sitting with the strangeness. Nothing is hot, nothing attracts, nothing repels — and
yet the gas pushes hard enough to hold up stars, because arithmetic forbids it to rest.
Degeneracy pressure may be the purest force in physics: a pressure made of counting.

The next notebook warms the sea by its actual 0.4% ([§7.10](fermi-gas-finite-temperature.ipynb): the Sommerfeld corrections, the linear
heat capacity, Pauli's paramagnetism), and the one after follows the pressure to its greatest
stage — a dying star weighing Pauli against gravity ([§7.11](white-dwarfs-chandrasekhar.ipynb)). Then the sea meets a lattice ([§7.12](bloch-theorem-band-structure.ipynb)),
and the movement discovers why some seas cannot flow at all.

## Notebook summary

Movement III opens with the ground state of dense, light matter — built by counting, checked
against handbooks.

- **The Fermi sea** {eq}`eq-fermi-sea`: the $T = 0$ step fills the sphere $|k| \le k_F$;
  $\varepsilon_F = (\hbar^2/2m)(3\pi^2n)^{2/3}$ inverted and verified by quadrature; the Fermi
  surface named as metal physics' organizing concept.
- **The sea, filled literally** {eq}`eq-literal-filling`: sorted box levels, spin-doubled
  *before* slicing (the $2^{2/3}$ trap stated); the $\varepsilon_F(N)$ shell **staircase**
  taught as finite-size physics (nuclear magic numbers' tabletop cousin); $E/(N\varepsilon_F) =
  0.638 \to 0.620 \to 0.609$, converging on $\tfrac35$ from above — the Weyl deficit, occupied
  side.
- **Metals by the numbers** {eq}`eq-metal-scales`: Cu $\varepsilon_F = 7.03$ eV, $T_F = 8.2
  \times10^4$ K, $v_F = 1.6\times10^6$ m/s; $T_{\text{room}}/T_F = 0.4$–$1.6\%$ — **metals live
  at absolute zero**, each electron carrying $\sim$100× the classical thermal energy as Pauli
  zero-point motion. The license of [§7.10](fermi-gas-finite-temperature.ipynb), written.
- **Degeneracy pressure** {eq}`eq-degeneracy-pressure`: $P = \tfrac25 n\varepsilon_F = 38$ GPa
  in copper — the $V^{-2/3}$ scaling exact in the literal filling and earned by finite
  differences in the continuum — the virial excess of [§7.8](classical-limit-thermal-wavelength.ipynb) grown into the stiffness of matter.
- **The data jewel** {eq}`eq-bulk-modulus`: $B = \tfrac23 n\varepsilon_F$ against the handbook —
  K $1.02$, Rb $0.91$, Cs $0.97$ with *nothing fitted*; Na's $1.46$ and Cu's $0.46$ taught as
  the model's honest boundary (electron–ion softening; the d-band).
- **The sound of Pauli** {eq}`eq-bohm-staver`: $v_s = v_F\sqrt{Zm_e/3M}$ — sodium $3011$ m/s vs
  $\sim3200$ measured: the exclusion principle, audible.
- **The cliff, located** {eq}`eq-relativistic-threshold`: $v_F = c$ at $n_{\text{rel}} =
  5.87\times10^{35}$ m⁻³ — bracketed by white-dwarf core densities (ratios $0.51$ and $5.1$);
  the pressure law bends from $n^{5/3}$ to $n^{4/3}$ exactly where the dead stars live. [§7.11](white-dwarfs-chandrasekhar.ipynb)
  walks off the edge.

A pressure made of counting; next, its thermal corrections, and then its greatest stage.

## Outlook

- **The sea, warmed by 0.4% ([§7.10](fermi-gas-finite-temperature.ipynb)).** $\mu(T)$, the linear heat capacity, Pauli paramagnetism —
  the Sommerfeld program, tooled in [§7.3](statistical-toolkit.ipynb).
- **The pressure's greatest stage ([§7.11](white-dwarfs-chandrasekhar.ipynb)).** White dwarfs and the Chandrasekhar mass — the
  relativistic gas developed in full.
- **The sea meets a lattice ([§7.12](bloch-theorem-band-structure.ipynb)).** Bloch, bands, and why some seas cannot flow; degenerate
  doping returns in [§7.13](semiconductor-statistics.ipynb).
- **Interacting electrons** (screening, Fermi-liquid theory): named horizons, Volume VIII
  territory.
- **Cross-reference** [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (Pauli, the sea's foundation), [§7.7](bose-einstein-fermi-dirac.ipynb) (the step), [§7.8](classical-limit-thermal-wavelength.ipynb) (the degeneracy
  map; the virial embryo), [§7.3](statistical-toolkit.ipynb) (the DOS, the counting, the Weyl deficit — twice reused here).

In [ ]:
from ecp.style import footer

footer()